# Fleet Simulation Backend — Aşama 2: Ajansız Tick Döngüsü

Amaç: 12-15 sentetik rulmanı aynı anda, sürekli ilerleten bir arka plan
döngüsü kurmak. Her makine kendi bağımsız ömür eğrisinde ilerliyor, ömrü
bitince otomatik olarak yeniden başlıyor. Sonuç, Supabase'deki `machines`
tablosuna yazılıyor. Henüz ajan/LLM/ekip mantığı yok.

In [40]:
from dotenv import load_dotenv
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

SUPABASE_URL = os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY")

print("URL bulundu mu:", SUPABASE_URL is not None)
print("Key bulundu mu:", SUPABASE_KEY is not None)

URL bulundu mu: True
Key bulundu mu: True


In [41]:
import os
import time
import random
import numpy as np
import pandas as pd
import pickle
from supabase import create_client, Client

# proje kökünden model dosyalarına erişim
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))  # notebook 'notebooks/' altındaysa
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')

with open(os.path.join(MODELS_DIR, 'bearing_model.pkl'), 'rb') as f:
    bearing_model = pickle.load(f)

avg_template = pd.read_csv(os.path.join(MODELS_DIR, 'bearing_avg_template.csv'))
std_template = pd.read_csv(os.path.join(MODELS_DIR, 'bearing_std_template.csv'))

feature_cols = ['rms_rm_norm', 'kurtosis_rm_norm']

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Model ve şablonlar yüklendi.")
print("Toplam ömür noktası (avg_template):", len(avg_template))

Model ve şablonlar yüklendi.
Toplam ömür noktası (avg_template): 1000


## Çoklu Makine State'i ve Tek Bir Tick Fonksiyonu

Her makine için: mevcut life_pct, oradan üretilen sensör değerleri (ortalama
şablon + gürültü), ve modelin ürettiği risk olasılığı. Bir makine %100'e
ulaşınca otomatik olarak sıfırdan (%0) yeniden başlıyor.

In [43]:
import numpy as np
import pandas as pd

N_MACHINES = 12
LIFE_STEP_PCT = 0.5   # her tick'te ömrün ne kadarı ilerliyor (ayarlanabilir)
NOISE_SCALE = 0.3

def init_machine_state(machine_id):
    return {"id": machine_id, "life_pct": np.random.uniform(0, 5)}  # hafif rastgele başlangıç

machines_state = {f"M{i+1:02d}": init_machine_state(f"M{i+1:02d}") for i in range(N_MACHINES)}
print("Başlangıç durumu:")
for m in machines_state.values():
    print(m)

Başlangıç durumu:
{'id': 'M01', 'life_pct': 2.0789212490820135}
{'id': 'M02', 'life_pct': 3.480249065360253}
{'id': 'M03', 'life_pct': 0.2581235485066946}
{'id': 'M04', 'life_pct': 0.2679892836865483}
{'id': 'M05', 'life_pct': 1.5721718979832406}
{'id': 'M06', 'life_pct': 1.4566535474226256}
{'id': 'M07', 'life_pct': 1.6356987310814715}
{'id': 'M08', 'life_pct': 1.3765130817005822}
{'id': 'M09', 'life_pct': 4.879084151629986}
{'id': 'M10', 'life_pct': 3.0650129930677146}
{'id': 'M11', 'life_pct': 3.6649920580323285}
{'id': 'M12', 'life_pct': 2.4625393521438923}


In [44]:
def get_features_at_life_pct(life_pct, avg_template, std_template, feature_cols, rng):
    idx = int(np.clip(life_pct / 100 * (len(avg_template) - 1), 0, len(avg_template) - 1))
    row = {}
    for col in feature_cols:
        mean_val = avg_template[col].iloc[idx]
        std_val = std_template[col].iloc[idx]
        row[col] = mean_val + rng.normal(0, std_val * NOISE_SCALE)
    return row

def tick_machine(state, avg_template, std_template, feature_cols, model, rng):
    state["life_pct"] += LIFE_STEP_PCT
    if state["life_pct"] >= 100:
        state["life_pct"] = 0.0  # ömür bitti, yeniden başla

    features = get_features_at_life_pct(state["life_pct"], avg_template, std_template, feature_cols, rng)
    X = pd.DataFrame([features])[feature_cols]
    risk_proba = model.predict_proba(X)[0, 1]

    status = "healthy"
    if risk_proba >= 0.75:
        status = "at_risk"

    return {
        "id": state["id"],
        "status": status,
        "risk_probability": float(risk_proba),
        "life_pct": float(state["life_pct"]),
        "top_shap_feature": None,  # ajan aşamasında dolduracağız
    }

# tek bir makinede test edelim
rng = np.random.default_rng(42)
result = tick_machine(machines_state["M01"], avg_template, std_template, feature_cols, bearing_model, rng)
print(result)

{'id': 'M01', 'status': 'healthy', 'risk_probability': 0.007833148842019447, 'life_pct': 2.5789212490820135, 'top_shap_feature': None}


## Tüm Makineleri İlerleten ve Supabase'e Yazan Döngü

Her turda 12 makinenin hepsi bir adım ilerliyor, sonuç Supabase'deki
`machines` tablosuna upsert ediliyor (satır zaten varsa günceller, yoksa
oluşturur).

## Sürekli Döngü Testi (Notebook İçinde, Kısa Süreli)

Backend'e taşımadan önce, birkaç tur art arda çalıştırıp makinelerin gerçekten
zamanla ilerlediğini (life_pct arttığını, bazılarının risk'e girip
sıfırlandığını) gözle doğruluyoruz.

## Makinelere Bireysel "Aşınma Kişiliği" Ekleme

Her makineye başlangıçta sabit, rastgele bir aşınma hızı çarpanı ve baskın
özellik ataması yapıyoruz — böylece risk artışının "neden"i gerçekten
izlenebilir ve tutarlı oluyor, SHAP çıktısı kurgusal değil anlamlı hale
geliyor.

In [ ]:
def init_machine_state_v2(machine_id, rng):
    wear_speed = rng.uniform(0.7, 1.4)  # bazıları yavaş, bazıları hızlı aşınıyor
    dominant_feature = rng.choice(feature_cols)  # hangi özellik bu makinede daha "aktif"
    dominant_boost = rng.uniform(1.3, 2.0)  # o özelliğin ne kadar abartılı yükseleceği

    return {
        "id": machine_id,
        "life_pct": rng.uniform(0, 5),
        "wear_speed": wear_speed,
        "dominant_feature": dominant_feature,
        "dominant_boost": dominant_boost,
    }

rng_init = np.random.default_rng(42)
machines_state_v2 = {f"M{i+1:02d}": init_machine_state_v2(f"M{i+1:02d}", rng_init) for i in range(N_MACHINES)}

for m in machines_state_v2.values():
    print(m)

{'id': 'M01', 'life_pct': 3.4868401452968194, 'wear_speed': 1.2417692339891742, 'dominant_feature': np.str_('kurtosis_rm_norm'), 'dominant_boost': 1.9010185439379677}
{'id': 'M02', 'life_pct': 3.805698509951765, 'wear_speed': 0.7659241435213546, 'dominant_feature': np.str_('rms_rm_norm'), 'dominant_boost': 1.9829356461457293}
{'id': 'M03', 'life_pct': 1.8539901211629062, 'wear_speed': 1.2502450136938676, 'dominant_feature': np.str_('kurtosis_rm_norm'), 'dominant_boost': 1.615270156526897}
{'id': 'M04', 'life_pct': 4.113808066354149, 'wear_speed': 1.3487354921940211, 'dominant_feature': np.str_('rms_rm_norm'), 'dominant_boost': 1.7507055840564651}
{'id': 'M05', 'life_pct': 0.3190862805208766, 'wear_speed': 1.0103899391791318, 'dominant_feature': np.str_('rms_rm_norm'), 'dominant_boost': 1.6882093509110845}
{'id': 'M06', 'life_pct': 3.7904387004268694, 'wear_speed': 1.2793418203948073, 'dominant_feature': np.str_('rms_rm_norm'), 'dominant_boost': 1.7421650793854455}
{'id': 'M07', 'life_p

## tick_machine Fonksiyonunu Güncelleme: Kişiliği Kullanmak

Aşınma hızı artık `wear_speed` ile ölçekleniyor. Baskın özellik, ortalama
şablon değerine `dominant_boost` ile çarpılan ekstra bir sapma ekleniyor —
böylece o makinede gerçekten o özellik daha güçlü/erken yükseliyor, ve SHAP
bunu tutarlı biçimde yakalayacak.

In [ ]:
def get_features_at_life_pct_v2(life_pct, avg_template, std_template, feature_cols,
                                  dominant_feature, dominant_boost, rng):
    idx = int(np.clip(life_pct / 100 * (len(avg_template) - 1), 0, len(avg_template) - 1))
    row = {}
    for col in feature_cols:
        mean_val = avg_template[col].iloc[idx]
        std_val = std_template[col].iloc[idx]
        base_noise = rng.normal(0, std_val * NOISE_SCALE)

        if col == dominant_feature:
            # baskın özellik: ortalamadan sapmayı abart (mean'in üzerindeki kısmı büyüt)
            deviation_from_baseline = mean_val - avg_template[col].iloc[0]
            boosted_value = avg_template[col].iloc[0] + deviation_from_baseline * dominant_boost
            row[col] = boosted_value + base_noise
        else:
            row[col] = mean_val + base_noise
    return row

def tick_machine_v2(state, avg_template, std_template, feature_cols, model, rng):
    state["life_pct"] += LIFE_STEP_PCT * state["wear_speed"]
    if state["life_pct"] >= 100:
        state["life_pct"] = 0.0
        # yeniden başlarken kişiliği de yenileyelim (yeni bir "birim" gibi)
        state["wear_speed"] = rng.uniform(0.7, 1.4)
        state["dominant_feature"] = rng.choice(feature_cols)
        state["dominant_boost"] = rng.uniform(1.3, 2.0)

    features = get_features_at_life_pct_v2(
        state["life_pct"], avg_template, std_template, feature_cols,
        state["dominant_feature"], state["dominant_boost"], rng
    )
    X = pd.DataFrame([features])[feature_cols]
    risk_proba = model.predict_proba(X)[0, 1]

    status = "healthy"
    if risk_proba >= 0.75:
        status = "at_risk"

    return {
        "id": state["id"],
        "status": status,
        "risk_probability": float(risk_proba),
        "life_pct": float(state["life_pct"]),
        "top_shap_feature": state["dominant_feature"],  # şimdilik doğrudan kişilikten,
                                                          # gerçek SHAP hesabı Tanı Ajanında (Aşama 3) gelecek
    }

# hızlı test: bir makineyi ömrünün geç bir noktasına manuel taşıyıp risk/dominant_feature tutarlılığına bakalım
test_state = {"id": "TEST", "life_pct": 90.0, "wear_speed": 1.0,
              "dominant_feature": "kurtosis_rm_norm", "dominant_boost": 1.9}
rng_test = np.random.default_rng(1)
for _ in range(5):
    r = tick_machine_v2(test_state, avg_template, std_template, feature_cols, bearing_model, rng_test)
    print(r)

{'id': 'TEST', 'status': 'healthy', 'risk_probability': 0.022513992045840675, 'life_pct': 90.5, 'top_shap_feature': 'kurtosis_rm_norm'}
{'id': 'TEST', 'status': 'healthy', 'risk_probability': 0.35248813892068137, 'life_pct': 91.0, 'top_shap_feature': 'kurtosis_rm_norm'}
{'id': 'TEST', 'status': 'healthy', 'risk_probability': 0.1599000368685636, 'life_pct': 91.5, 'top_shap_feature': 'kurtosis_rm_norm'}
{'id': 'TEST', 'status': 'healthy', 'risk_probability': 0.007173363625985877, 'life_pct': 92.0, 'top_shap_feature': 'kurtosis_rm_norm'}
{'id': 'TEST', 'status': 'healthy', 'risk_probability': 0.0402601175267091, 'life_pct': 92.5, 'top_shap_feature': 'kurtosis_rm_norm'}


In [ ]:
idx_90 = int(90 / 100 * (len(avg_template) - 1))
print("Ortalama şablonda %90 life_pct'teki değerler:")
print(avg_template.iloc[idx_90][feature_cols])
print("\nBaşlangıç (life_pct=0) değerleri:")
print(avg_template.iloc[0][feature_cols])

print("\nBoosted kurtosis hesabı (manuel):")
mean_val = avg_template['kurtosis_rm_norm'].iloc[idx_90]
baseline = avg_template['kurtosis_rm_norm'].iloc[0]
deviation = mean_val - baseline
boosted = baseline + deviation * 1.9
print(f"Orijinal ortalama: {mean_val:.3f}, boosted: {boosted:.3f}, sapma: {deviation:.3f}")

# std_template'teki gürültü seviyesiyle karşılaştıralım - boost, gürültünün içinde kayboluyor olabilir
std_val_90 = std_template['kurtosis_rm_norm'].iloc[idx_90]
print(f"\n%90'daki std (gürültü seviyesi): {std_val_90:.3f}")
print(f"Bizim eklediğimiz gürültü (std * {NOISE_SCALE}): {std_val_90 * NOISE_SCALE:.3f}")

Ortalama şablonda %90 life_pct'teki değerler:
rms_rm_norm         1.202262
kurtosis_rm_norm    2.383680
Name: 899, dtype: float64

Başlangıç (life_pct=0) değerleri:
rms_rm_norm         1.000142
kurtosis_rm_norm    1.501785
Name: 0, dtype: float64

Boosted kurtosis hesabı (manuel):
Orijinal ortalama: 2.384, boosted: 3.177, sapma: 0.882

%90'daki std (gürültü seviyesi): 3.678
Bizim eklediğimiz gürültü (std * 0.3): 1.103
